[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fedgutcor/lab-ia-intermedio/blob/main/recetas/04_evaluar_guardar_servir.ipynb)

# 04 · Evaluar, registrar, guardar y servir un modelo

**M4 · Evaluación, versionado y despliegue** · Laboratorio de IA · nivel intermedio

En esta receta vas a:
- comparar una línea base con una variante bajo el mismo protocolo
- leer métricas por clase y una matriz de confusión
- registrar cada experimento con sus parámetros
- guardar el modelo, cargarlo y usarlo con un contrato de entrada y salida

**Cómo usarla:** en Colab elige *Entorno de ejecución → Ejecutar todo*. No hace falta instalar nada. Si Colab avisa que el notebook no es de Google, elige *Ejecutar de todos modos*. Cuando termine, busca la celda **«Cambia esto»**, modifica un valor y vuelve a ejecutar.

In [ ]:
import time, json, datetime, os
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
def como_texto(x):
    # Keras 3 recibe el texto como tensor de TensorFlow, no como columna de pandas
    return tf.constant([str(t) for t in x])
import matplotlib.pyplot as plt

INICIO = time.time()
keras.utils.set_random_seed(0)
df = pd.read_csv("https://raw.githubusercontent.com/Fedgutcor/lab-ia-intermedio/main/datos/opiniones.csv")
print(df["sentimiento"].value_counts(), "\n")
print(df["dificultad"].value_counts())

## 1 · Un solo protocolo para todo

Se parte **por frase** una sola vez: la prueba trae opiniones con frases que los modelos nunca vieron. Todos los modelos se evalúan con esa misma prueba.

In [ ]:
CLASES = ["negativa", "positiva"]
df["y"] = (df["sentimiento"] == "positiva").astype(int)
i_ent, i_pru = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0).split(df, groups=df["frase"]))
ent, pru = df.iloc[i_ent], df.iloc[i_pru]
print(f"Entrenamiento: {len(ent)} · Prueba: {len(pru)} · Frases en prueba: {pru['frase'].nunique()}")

## 2 · Cambia esto

In [ ]:
NGRAMAS = 2        # 1 = palabras sueltas · 2 = también pares («no recomiendo», «nada mal»)
EPOCAS = 20

## 3 · Línea base y variante

In [ ]:
REGISTRO = "registro_experimentos.csv"

def registrar(nombre, params, metricas):
    fila = {"fecha": datetime.datetime.now().isoformat(timespec="seconds"), "modelo": nombre,
            "parametros": json.dumps(params, ensure_ascii=False), **metricas}
    pd.DataFrame([fila]).to_csv(REGISTRO, mode="a", header=not os.path.exists(REGISTRO), index=False)

def metricas(y_real, y_pred, dificultad):
    m = {"exactitud": round(float((y_real == y_pred).mean()), 3)}
    for d in ["normal", "negación", "ironía"]:
        sel = dificultad == d
        m[f"exactitud_{d}"] = round(float((y_real[sel] == y_pred[sel]).mean()), 3) if sel.any() else None
    return m

# línea base: predecir siempre la clase más frecuente del entrenamiento
mayoritaria = int(ent["y"].mode()[0])
pred_base = np.full(len(pru), mayoritaria)
m_base = metricas(pru["y"].values, pred_base, pru["dificultad"].values)
registrar("linea_base_mayoritaria", {"clase": CLASES[mayoritaria]}, m_base)

# variante: bolsa de palabras (y pares) + una capa densa
# output_mode="count": en Keras 3.13 (la versión de Colab) un modelo con "tf_idf" se guarda,
# pero load_model no lo puede volver a cargar. Una dependencia de versión real, comprobada.
vectorizar = layers.TextVectorization(max_tokens=5000, ngrams=NGRAMAS, output_mode="count")
vectorizar.adapt(ent["opinion"].values)
modelo = keras.Sequential([keras.Input((1,), dtype="string"), vectorizar, layers.Dense(2, activation="softmax")])
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.fit(como_texto(ent["opinion"]), ent["y"].values, epochs=EPOCAS, batch_size=32, verbose=0)
pred_var = modelo.predict(como_texto(pru["opinion"]), verbose=0).argmax(axis=1)
m_var = metricas(pru["y"].values, pred_var, pru["dificultad"].values)
registrar("bolsa_densa", {"ngramas": NGRAMAS, "epocas": EPOCAS}, m_var)

pd.read_csv(REGISTRO)

## 4 · Métricas por clase y matriz de confusión

Un promedio puede esconder la clase que peor se atiende. Mira cada clase por separado.

In [ ]:
print(classification_report(pru["y"], pred_var, target_names=CLASES, digits=3, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(pru["y"], pred_var), display_labels=CLASES).plot(cmap="Blues")
plt.title("Variante · prueba"); plt.show()

## 5 · Guardar, cargar y servir

Guardar el modelo es una cosa; poder consultarlo con reglas claras es otra. La función `predecir` es un **contrato**: dice qué acepta, qué devuelve y qué hace con una entrada inválida.

In [ ]:
modelo.save("modelo_opiniones.keras")
cargado = keras.models.load_model("modelo_opiniones.keras")
iguales = np.allclose(modelo.predict(como_texto(pru["opinion"][:20]), verbose=0), cargado.predict(como_texto(pru["opinion"][:20]), verbose=0))
print("¿El modelo cargado predice lo mismo que el original?", iguales)

VERSION = "opiniones-" + datetime.date.today().isoformat()

def predecir(texto):
    if not isinstance(texto, str) or not texto.strip():
        return {"error": "La entrada debe ser un texto no vacío."}
    if len(texto) > 500:
        return {"error": "El texto supera 500 caracteres."}
    p = cargado.predict(como_texto([texto]), verbose=0)[0]
    return {"sentimiento": CLASES[int(p.argmax())], "confianza": round(float(p.max()), 3), "version": VERSION}

for prueba_txt in ["Llegó rápido y funciona perfecto", "No está nada mal", "Excelente: llegó roto.", "", 42]:
    print(repr(prueba_txt), "→", predecir(prueba_txt))
print(f"\nTiempo total de la receta: {time.time() - INICIO:.0f} s")

## 6 · Bonus: una interfaz web con Gradio

Con Gradio (viene en Colab) el modelo se usa desde un formulario web, sin escribir código. Es un despliegue **de ensayo**: sirve para mostrar y probar, no es un servicio en producción.

In [ ]:
import sys
import gradio as gr

demo = gr.Interface(
    fn=predecir,
    inputs=gr.Textbox(label="Escribe una opinión"),
    outputs=gr.JSON(label="Respuesta del modelo"),
    title="Clasificador de opiniones",
    examples=["Llegó rápido y funciona perfecto", "No está nada mal", "Excelente: llegó roto."],
)
if "google.colab" in sys.modules:
    demo.launch()      # la interfaz aparece debajo de esta celda
else:
    print("Interfaz lista. En Colab aparece al ejecutar esta celda; en tu equipo usa demo.launch().")

## Opcional · seguimiento con MLflow

El archivo `registro_experimentos.csv` ya cumple la función de registro. Si quieres una interfaz de seguimiento, MLflow funciona en Colab instalándolo:

```python
!pip install -q mlflow
import mlflow
with mlflow.start_run(run_name="bolsa_densa"):
    mlflow.log_params({"ngramas": NGRAMAS, "epocas": EPOCAS})
    mlflow.log_metrics({k: v for k, v in m_var.items() if v is not None})
```

## Para tu proyecto

- Si la variante gana en promedio pero pierde en la clase que más le importa al usuario, la decisión no es automática.
- Guarda siempre **qué versión** produjo cada predicción.